In [5]:
import pandas as pd

df = pd.read_excel("./data/data.xlsx")

In [15]:
import time
import asyncpg

db_name="BunkeringBot"
db_user="def_user"
db_password="super_password"
db_host="77.37.96.222"
db_port="5432"

pool = await asyncpg.create_pool(
            database=db_name,
            user=db_user,
            password=db_password,
            host=db_host,
            port=db_port,
            min_size=1,
            max_size=20
        )

In [13]:
data = df[["UN/LOCODE", "Group number"]].copy()
data = data[data['Group number'].notna()]

data

,UN/LOCODE,Group number
1743,MYPKG,1.0
2268,SGSIN,1.0
2353,CNZOS,2.0
2356,HKHKG,2.0
3309,KRPUS,2.0
3725,TWKHH,2.0
3830,THBKK,3.0
3903,VNSGN,3.0
4565,OMSOH,4.0
5203,AEJEA,4.0


In [17]:
records = [
        (row["Group number"], row["UN/LOCODE"])
        for _, row in data.iterrows()
    ]

async with pool.acquire() as conn:
    await conn.executemany(
        """
        INSERT INTO public.port_group (group_id, port_locode)
        VALUES ($1, $2)
        """,
        records
    )

await pool.close()